In [ ]:
import time, os 
import requests as http_requests

NOTEBOOK_NAME = "anime-omni-assemble"
STEP_NAME = "step_omni"

# Carregar secrets
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    def _ks(n):
        try: return _s.get_secret(n)
        except: return ""
    DATABASE_URL = _ks("DATABASE_URL")
    PROJECT_ID = _ks("PIPELINE_PROJECT_ID")
    PIPELINE_WEBHOOK_URL = _ks("PIPELINE_WEBHOOK_URL")
    TASK_KEY = _ks("PIPELINE_TASK_KEY")
except:
    DATABASE_URL = os.getenv("DATABASE_URL", "")
    PROJECT_ID = os.getenv("PIPELINE_PROJECT_ID", "")
    PIPELINE_WEBHOOK_URL = os.getenv("PIPELINE_WEBHOOK_URL", "")
    TASK_KEY = os.getenv("PIPELINE_TASK_KEY", os.getenv("TASK_KEY", ""))

if not TASK_KEY:
    TASK_KEY = "omni-assemble"

def should_run(step):
    """Decide se um passo (translate, tts, assemble) deve ser executado."""
    if TASK_KEY in ["omni", "omni-main", ""]:
        return True
    if step == "translate":
        return TASK_KEY in ["omni", "omni-main", ""]
    if step == "tts":
        return TASK_KEY in ["omni", "omni-main", ""] or TASK_KEY.startswith("omni-tts")
    if step == "assemble":
        return TASK_KEY in ["omni", "omni-main", "omni-assemble"]
    return True

_cell_timers = {}

def _db_exec(query, params):
    if not DATABASE_URL: return
    try:
        import psycopg2
        conn = psycopg2.connect(DATABASE_URL)
        cur = conn.cursor()
        cur.execute(query, params)
        conn.commit()
        cur.close()
        conn.close()
    except: pass

def cell_start(idx, name=""):
    _cell_timers[idx] = time.time()
    print(f"\n{'='*50}\n  CELULA [{idx}] {name}\n{'='*50}")
    if not PROJECT_ID: return
    _db_exec("INSERT INTO pipeline_cell_tracking (project_id,notebook,cell_index,cell_name,status,started_at) VALUES (%s::uuid,%s,%s,%s,'running',NOW()) ON CONFLICT DO NOTHING", (PROJECT_ID, NOTEBOOK_NAME, idx, name))
    _db_exec("UPDATE pipeline_cell_tracking SET status='running',started_at=NOW(),finished_at=NULL,cell_name=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (name, PROJECT_ID, NOTEBOOK_NAME, idx))

def cell_end(idx, status="done", msg=""):
    elapsed = ""
    if idx in _cell_timers:
        s = int(time.time() - _cell_timers.pop(idx))
        elapsed = f" ({s//60}m{s%60}s)" if s >= 60 else f" ({s}s)"
    icon = "OK" if status == "done" else "ERRO"
    print(f"  [{icon}] CELULA [{idx}] {status}{elapsed}: {msg}\n{'_'*50}")
    if not PROJECT_ID: return
    _db_exec("UPDATE pipeline_cell_tracking SET status=%s,finished_at=NOW(),duration_seconds=EXTRACT(EPOCH FROM(NOW()-started_at)),message=%s WHERE project_id=%s::uuid AND notebook=%s AND cell_index=%s", (status, msg, PROJECT_ID, NOTEBOOK_NAME, idx))

def report_step(status, msg=""):
    if PROJECT_ID and PIPELINE_WEBHOOK_URL:
        try:
            http_requests.post(f"{PIPELINE_WEBHOOK_URL}/webhook/status", json={"project_id": PROJECT_ID, "step": STEP_NAME, "status": status, "message": msg}, timeout=15)
        except: pass
    print(f"\nNOTEBOOK FINALIZADO: {STEP_NAME} -> {status}")
    if not PROJECT_ID: return
    _db_exec(f"UPDATE pipeline_projects SET {STEP_NAME}=%s,current_step=%s,updated_at=NOW() WHERE id=%s::uuid", (status, STEP_NAME.replace("step_",""), PROJECT_ID))
    _db_exec("INSERT INTO pipeline_logs (project_id,step,status,message) VALUES (%s::uuid,%s,%s,%s)", (PROJECT_ID, STEP_NAME, status, msg))

def fetch_project_opts():
    if not DATABASE_URL or not PROJECT_ID: return {"bg_audio": False, "srt_type": "normal", "azure_enabled": True}
    try:
        import psycopg2
        from psycopg2.extras import RealDictCursor
        conn = psycopg2.connect(DATABASE_URL, cursor_factory=RealDictCursor)
        cur = conn.cursor()
        cur.execute("SELECT bg_audio, srt_type, azure_enabled FROM pipeline_projects WHERE id=%s::uuid", (PROJECT_ID,))
        row = cur.fetchone()
        cur.close()
        conn.close()
        return dict(row) if row else {"bg_audio": False, "srt_type": "normal", "azure_enabled": True}
    except Exception as e:
        print(f"Erro ao buscar opts: {e}")
        return {"bg_audio": False, "srt_type": "normal", "azure_enabled": True}

PROJECT_OPTS = fetch_project_opts()
AZURE_ENABLED = PROJECT_OPTS.get("azure_enabled", True)
print("Tracking de celulas e opts configurados!", PROJECT_OPTS)


In [ ]:
cell_start(1, "Celula 1")
# @title 🚀 1. Setup Inicial: Kaggle + Google Drive OAuth + Whisper + Modelos v6
# @markdown Configure TUDO aqui antes de rodar qualquer outra célula.

import os, sys, shutil, time, asyncio, json, nest_asyncio, subprocess, urllib.request, io, gc
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

# --- 1. INSTALAÇÃO DE DEPENDÊNCIAS ---
print("📦 Instalando bibliotecas (pode demorar 2-3 min)...")
os.system("apt-get install -y ffmpeg > /dev/null 2>&1")
os.system("pip install python-dotenv faster-whisper gradio_client google-genai edge-tts ffmpeg-python pydub openai nest_asyncio omnivoice torchaudio google-auth google-auth-httplib2 google-api-python-client stable-ts funasr modelscope demucs silero-vad onnxruntime pyannote.audio speechbrain scipy > /dev/null 2>&1")

import torch
import stable_whisper
from google import genai as google_genai_client
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from openai import OpenAI
from pydub import AudioSegment
from dotenv import load_dotenv
nest_asyncio.apply()

# ─────────────────────────────────────────────────────────────────────────────
# ⚙️  PAINEL CENTRAL DE CONFIGURAÇÃO — Edite aqui antes de rodar
# ─────────────────────────────────────────────────────────────────────────────



# 🔊 AJUSTE DE VELOCIDADE DE ÁUDIO (Fábrica Cel 6)
TARGET_MIN_SPEED = 1.1
TARGET_MAX_SPEED = 1.35
MAX_AUDIO_RETRIES = 4
MAX_TEXT_RETRIES  = 3

# 🎙️ NARRADOR PADRÃO (usado para busca do áudio de clonagem no Drive)
# O sistema busca arquivos na pasta CLONAGEM/ do Drive cujo nome se pareça com esse.
# Ex: NOME_NARRADOR="Goku" vai achar "goku_dragon_ball.mp3" automaticamente.
NOME_NARRADOR = "Alessandro"  # @param {type:"string"}

# 📦 TRADUÇÃO — TAMANHO DO BATCH (blocos por chamada ao modelo)
# Se o anime foi identificado: usa BATCH_SIZE. Senão: usa 60 (mais conservador).
BATCH_SIZE = 50  # @param {type:"integer"}

# ─────────────────────────────────────────────────────────────────────────────
# 🤖 CADEIAS DE MODELOS GEMINI (fallback automático por quota/erro)
#    Adicione modelos extras separados por vírgula — serão tentados em ordem.
# ─────────────────────────────────────────────────────────────────────────────

# Identificação de anime — requer modelo com boa compreensão multilíngue
MODELS_IDENTIFICACAO = [
    "gemini-3.5-flash",
    "gemini-3.1-pro-preview",
    "gemini-3.1-flash-lite",
    "gemini-3-flash-preview",
    "gemini-2.5-pro"
]

# Tradução em batch — velocidade é importante, flash é suficiente
MODELS_TRADUCAO = [
    "gemini-3.5-flash",
    "gemini-3.1-pro-preview",
    "gemini-3.1-flash-lite",
    "gemini-2.5-pro",
    "gemini-3-flash-preview"
]


# Reescrita de texto da fábrica de áudio — contagem de caracteres
MODELS_REESCRITA = [
    "gemini-3.5-flash",
    "gemini-3.1-flash-lite",
    "gemini-3-flash-preview"
]

# Fallback para OpenAI caso TODOS os Gemini falhem
OPENAI_FALLBACK_MODEL = "gpt-5.4"
USE_OPENAI_FALLBACK   = True  # @param {type:"boolean"}

# ─────────────────────────────────────────────────────────────────────────────
# 🔑 CHAVES DE API — Kaggle Secrets (com fallback para .env local)
# ─────────────────────────────────────────────────────────────────────────────
def _load_credentials():
    # Tenta Kaggle Secrets primeiro (ambiente de produção)
    try:
        from kaggle_secrets import UserSecretsClient
        _s = UserSecretsClient()
        def _get(name):
            try: return _s.get_secret(name)
            except: return ""
        print("Carregando chaves do Kaggle Secrets...")
        return _get
    except ImportError:
        pass

    # Fallback: .env local (ambiente de desenvolvimento)
    load_dotenv()
    print("Carregando chaves do .env (desenvolvimento local)...")
    return lambda name: os.getenv(name, "")

_get_secret = _load_credentials()

GEMINI_API_KEY      = _get_secret("GEMINI_API_KEY")
OPENAI_API_KEY      = _get_secret("OPENAI_API_KEY")
DRIVE_ACCESS_TOKEN  = _get_secret("DRIVE_ACCESS_TOKEN")
DRIVE_REFRESH_TOKEN = _get_secret("DRIVE_REFRESH_TOKEN")
DRIVE_CLIENT_ID     = _get_secret("DRIVE_CLIENT_ID")
DRIVE_CLIENT_SECRET = _get_secret("DRIVE_CLIENT_SECRET")
HF_TOKEN            = _get_secret("HF_TOKEN")
AZURE_OPENAI_ENDPOINT   = _get_secret("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY    = _get_secret("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_DEPLOYMENT = _get_secret("AZURE_OPENAI_DEPLOYMENT")

_keys_ok = True
for _name, _val in [("GEMINI_API_KEY", GEMINI_API_KEY), ("DRIVE_REFRESH_TOKEN", DRIVE_REFRESH_TOKEN)]:
    if not _val:
        print(f"  ATENCAO: '{_name}' nao encontrada! Verifique Kaggle Secrets.")
        _keys_ok = False
if _keys_ok:
    print("Todas as chaves carregadas com sucesso.")

# ─────────────────────────────────────────────────────────────────────────────
# 📁 CAMINHOS DO PROJETO
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH            = "/kaggle/working/AUDIO_DUB"
TEMP_PATH            = "/kaggle/working/temp_audio"
ANIME_AUDIO_ORIGINAL = "/kaggle/working/input/anime_audio.mp3"
MIN_GAP_CENA_S       = 0.3

MODELOS_OFFLINE       = ""
MODEL_WHISPER_PATH    = "large-v3"
# SenseVoice: prioridade de carregamento
# 1. Dataset Kaggle local (instantâneo)
# 2. HuggingFace (CDN global, rápido)
# 3. ModelScope (servidores China, lento - último recurso)
# Tenta encontrar o SenseVoiceSmall no /kaggle/input de forma case-insensitive
import os as _os
SENSEVOICE_MODEL_PATH = None
if _os.path.exists("/kaggle/input"):
    for _root, _dirs, _files in _os.walk("/kaggle/input"):
        for _d in _dirs:
            if _d.lower() == "sensevoicesmall":
                SENSEVOICE_MODEL_PATH = _os.path.join(_root, _d)
                print(f"[SenseVoice] Encontrado no dataset Kaggle: {SENSEVOICE_MODEL_PATH}")
                break
        if SENSEVOICE_MODEL_PATH:
            break

if not SENSEVOICE_MODEL_PATH:
    # Se não achou localmente, tenta baixar do HuggingFace (muito mais rápido que ModelScope)
    try:
        from huggingface_hub import snapshot_download
        _sv_hf = snapshot_download("FunAudioLLM/SenseVoiceSmall",
                                   local_dir="/kaggle/working/SenseVoiceSmall",
                                   token=HF_TOKEN if HF_TOKEN else None)
        SENSEVOICE_MODEL_PATH = _sv_hf
        print(f"[SenseVoice] Baixado do HuggingFace: {_sv_hf}")
    except Exception as _hf_err:
        print(f"[SenseVoice] HuggingFace falhou ({_hf_err}), usando ModelScope...")
        SENSEVOICE_MODEL_PATH = "iic/SenseVoiceSmall"
        print(f"[SenseVoice] Fallback ModelScope: {SENSEVOICE_MODEL_PATH}")
MODEL_OMNIVOICE_PATH  = "k2-fsa/OmniVoice"

os.chdir("/kaggle/working")
for p in [BASE_PATH, TEMP_PATH, "/kaggle/working/input"]:
    os.makedirs(p, exist_ok=True)

print("Limpando arquivos temporarios...")
for f in os.listdir("/kaggle/working"):
    if f.endswith((".mp3", ".wav", ".srt")):
        try: os.remove(f"/kaggle/working/{f}")
        except: pass

# ─────────────────────────────────────────────────────────────────────────────
# ☁️  GOOGLE DRIVE
# ─────────────────────────────────────────────────────────────────────────────
print("Autenticando Google Drive...")
try:
    creds = Credentials(
        token=DRIVE_ACCESS_TOKEN if DRIVE_ACCESS_TOKEN else None,
        refresh_token=DRIVE_REFRESH_TOKEN,
        token_uri="https://oauth2.googleapis.com/token",
        client_id=DRIVE_CLIENT_ID,
        client_secret=DRIVE_CLIENT_SECRET,
        scopes=["https://www.googleapis.com/auth/drive"]
    )
    if creds.expired and creds.refresh_token:
        creds.refresh(Request())
    drive_service = build("drive", "v3", credentials=creds)
    print("Google Drive autenticado!")
except Exception as e:
    drive_service = None
    print(f"Falha na autenticacao do Drive: {e}")

def _buscar_id(caminho_no_drive):
    partes = caminho_no_drive.strip("/").split("/")
    parent_id = "root"
    for parte in partes:
        query = f"name='{parte}' and '{parent_id}' in parents and trashed=false"
        results = drive_service.files().list(q=query, fields="files(id, mimeType)").execute()
        arquivos = results.get("files", [])
        if not arquivos: return None
        parent_id = arquivos[0]["id"]
    return parent_id

def _garantir_pasta(caminho_pasta):
    partes = caminho_pasta.strip("/").split("/")
    parent_id = "root"
    for pasta in partes:
        query = f"name='{pasta}' and '{parent_id}' in parents and trashed=false and mimeType='application/vnd.google-apps.folder'"
        results = drive_service.files().list(q=query, fields="files(id)").execute()
        existentes = results.get("files", [])
        if existentes:
            parent_id = existentes[0]["id"]
        else:
            nova = drive_service.files().create(
                body={"name": pasta, "mimeType": "application/vnd.google-apps.folder", "parents": [parent_id]},
                fields="id"
            ).execute()
            parent_id = nova["id"]
    return parent_id

def baixar_do_drive(caminho_no_drive, destino_local):
    if os.path.exists(destino_local): return True
    try:
        file_id = _buscar_id(caminho_no_drive)
        if not file_id: return False
        request = drive_service.files().get_media(fileId=file_id)
        with open(destino_local, "wb") as fh:
            downloader = MediaIoBaseDownload(fh, request)
            done = False
            while not done: _, done = downloader.next_chunk()
        return True
    except: return False

def salvar_no_drive(caminho_local, caminho_destino_drive):
    if drive_service is None or not os.path.exists(caminho_local): return
    try:
        partes = caminho_destino_drive.strip("/").split("/")
        nome_arquivo = partes[-1]
        pasta_drive  = "/".join(partes[:-1]) if len(partes) > 1 else ""
        parent_id = _garantir_pasta(pasta_drive) if pasta_drive else "root"
        query = f"name='{nome_arquivo}' and '{parent_id}' in parents and trashed=false"
        results = drive_service.files().list(q=query, fields="files(id)").execute()
        existentes = results.get("files", [])
        media = MediaFileUpload(caminho_local, resumable=True)
        if existentes:
            drive_service.files().update(fileId=existentes[0]["id"], media_body=media).execute()
        else:
            drive_service.files().create(
                body={"name": nome_arquivo, "parents": [parent_id]},
                media_body=media, fields="id"
            ).execute()
        print(f"  Salvo no Drive: {caminho_destino_drive}")
    except Exception as e:
        print(f"  Erro ao salvar {caminho_destino_drive}: {e}")

# Download de cache do Drive
if drive_service:
    print("\nBaixando arquivos do Drive...")
    baixar_do_drive("KAGGLE/AUDIO_DUB/INPUT/anime_audio.mp3",      "/kaggle/working/input/anime_audio.mp3")
    baixar_do_drive("KAGGLE/AUDIO_DUB/transcricao_raw.json",        f"{BASE_PATH}/transcricao_raw.json")
    baixar_do_drive("KAGGLE/AUDIO_DUB/traducao_simplificada.json",  f"{BASE_PATH}/traducao_simplificada.json")
    baixar_do_drive("KAGGLE/AUDIO_DUB/identificacao_anime.json",    f"{BASE_PATH}/identificacao_anime.json")

# --- Busca e download automático do áudio de referência para clonagem ---
REF_AUDIO_PATH = ""
REF_TEXT       = ""

if drive_service and NOME_NARRADOR:
    from difflib import SequenceMatcher
    def _sim(a, b): return SequenceMatcher(None, a.lower(), b.lower()).ratio()

    _clonagem_drive = "KAGGLE/AUDIO_DUB/INPUT/CLONAGEM"
    _clonagem_local = f"{BASE_PATH}/INPUT/CLONAGEM"
    os.makedirs(_clonagem_local, exist_ok=True)

    print(f"Buscando audio de clonagem para '{NOME_NARRADOR}'...")
    _pid = _buscar_id(_clonagem_drive)
    if _pid:
        _res = drive_service.files().list(
            q=f"'{_pid}' in parents and trashed=false and mimeType contains 'audio/'",
            fields="files(id, name)"
        ).execute()
        _melhor, _score = None, 0.0
        for _arq in _res.get('files', []):
            _s = _sim(NOME_NARRADOR, os.path.splitext(_arq['name'])[0])
            if NOME_NARRADOR.lower() in _arq['name'].lower(): _s = max(_s, 0.85)
            if _s > _score: _score, _melhor = _s, _arq
        if _melhor and _score > 0.6:
            print(f"  Match: '{_melhor['name']}' ({_score*100:.1f}%)")
            _local = f"{_clonagem_local}/{_melhor['name']}"
            if not os.path.exists(_local):
                _req = drive_service.files().get_media(fileId=_melhor['id'])
                with open(_local, "wb") as _f:
                    _dl = MediaIoBaseDownload(_f, _req)
                    _done = False
                    while not _done: _, _done = _dl.next_chunk()
            REF_AUDIO_PATH = _local
            print(f"  Referencia de clonagem definida: {REF_AUDIO_PATH}")
        else:
            print(f"  Nenhum match >60% para '{NOME_NARRADOR}' na pasta CLONAGEM.")
    else:
        print("  Pasta CLONAGEM nao encontrada no Drive.")

# ─────────────────────────────────────────────────────────────────────────────
# 🤖 CLIENTES DE API
# ─────────────────────────────────────────────────────────────────────────────
client_gemini = google_genai_client.Client(api_key=GEMINI_API_KEY) if GEMINI_API_KEY.startswith("AIza") else None
client_openai = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY.startswith("sk-") else None
client_azure = None
if AZURE_OPENAI_API_KEY and AZURE_OPENAI_ENDPOINT:
    print("Inicializando cliente Azure OpenAI...")
    client_azure = OpenAI(base_url=AZURE_OPENAI_ENDPOINT, api_key=AZURE_OPENAI_API_KEY)

# ─────────────────────────────────────────────────────────────────────────────
# 🔁 FUNÇÃO CENTRAL DE CHAMADA GEMINI COM FALLBACK POR QUOTA
# ─────────────────────────────────────────────────────────────────────────────
# Erros que indicam quota esgotada ou sobrecarga — deve tentar próximo modelo
QUOTA_ERROR_CODES = {429, 503, 500}
QUOTA_ERROR_MSGS  = [
    "quota", "rate limit", "resource exhausted", "overloaded",
    "too many requests", "service unavailable", "capacity",
    "not found", "not supported", "is not found", "timeout", "408", "deadline",
    "503", "500", "internal", "temporarily", "unavailable", "backoff",
]

def _is_quota_error(e: Exception) -> bool:
    msg = str(e).lower()
    if "psycopg" in msg or "connection" in msg or "port" in msg:
        return False
    code = getattr(e, "code", getattr(e, "status_code", None))
    if code in QUOTA_ERROR_CODES: return True
    return any(m in msg for m in QUOTA_ERROR_MSGS)

def azure_generate(prompt, response_format=None, system_instruction=None):
    if not client_azure:
        raise RuntimeError("client_azure nao inicializado")
    messages = []
    if system_instruction:
        messages.append({"role": "system", "content": str(system_instruction)})
    messages.append({"role": "user", "content": str(prompt)})
    try:
        kwargs = {
            "model": AZURE_OPENAI_DEPLOYMENT,
            "messages": messages,
            "temperature": 1.0
        }
        if response_format:
            kwargs["response_format"] = response_format
        resp = client_azure.chat.completions.create(**kwargs)
        return resp.choices[0].message.content
    except Exception as e:
        print(f"  [AZURE] Falha com chat.completions ({e}). Tentando responses.create...")
        resp = client_azure.responses.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            input=prompt
        )
        return resp.output[0]

def gemini_generate(model_chain: list, contents, config: dict = None, task_name: str = ""):
    """
    Tenta gerar conteúdo pelo Gemini (ou Azure OpenAI fallback se habilitado e primeiro falhar).
    """
    global AZURE_ENABLED
    if client_gemini is None:
        raise RuntimeError("client_gemini nao inicializado — verifique GEMINI_API_KEY no .env")

    # 1. Tentar primeiro modelo (geralmente gemini-3.5-flash)
    first_model = model_chain[0]
    try:
        kwargs = {"model": first_model, "contents": contents}
        if config:
            kwargs["config"] = config
        response = client_gemini.models.generate_content(**kwargs)
        print(f"  [OK] {task_name or 'gemini_generate'} -> {first_model}")
        return response
    except Exception as e:
        print(f"  [FAIL] {first_model} falhou: {str(e)[:100]}")
        
        # 2. Se habilitado no bot, tenta Azure OpenAI como segundo na prioridade
        if AZURE_ENABLED and client_azure:
            print(f"  [AZURE] Tentando Azure OpenAI ({AZURE_OPENAI_DEPLOYMENT}) como fallback...")
            try:
                sys_inst = None
                if config and "system_instruction" in config:
                    sys_inst = config["system_instruction"]
                resp_fmt = None
                if config and config.get("response_mime_type") == "application/json":
                    resp_fmt = {"type": "json_object"}
                
                azure_text = azure_generate(contents, resp_fmt, sys_inst)
                print(f"  [OK] {task_name or 'gemini_generate'} -> Azure ({AZURE_OPENAI_DEPLOYMENT})")
                
                class FakeGeminiResponse:
                    def __init__(self, text):
                        self.text = text
                return FakeGeminiResponse(azure_text)
            except Exception as ae:
                print(f"  [AZURE FAIL] Azure OpenAI falhou: {ae}")
        
        # 3. Prossegue com a cadeia padrão do Gemini
        last_exc = e
        for model in model_chain[1:]:
            try:
                kwargs = {"model": model, "contents": contents}
                if config:
                    kwargs["config"] = config
                response = client_gemini.models.generate_content(**kwargs)
                print(f"  [OK] {task_name or 'gemini_generate'} -> {model}")
                return response
            except Exception as e2:
                if _is_quota_error(e2):
                    print(f"  [QUOTA] {model}: {str(e2)[:80]} — tentando proximo...")
                    last_exc = e2
                    time.sleep(2)
                    continue
                else:
                    raise

        # 4. Fallback final para OpenAI se habilitado
        if USE_OPENAI_FALLBACK and client_openai:
            print(f"  [FALLBACK] Todos os modelos Gemini/Azure falharam. Usando OpenAI {OPENAI_FALLBACK_MODEL}...")
            raise RuntimeError(
                f"OPENAI_FALLBACK: Todos Gemini/Azure esgotados. "
                f"Ultimo erro: {last_exc}. "
                f"Use client_openai com modelo {OPENAI_FALLBACK_MODEL} no bloco catch da celula chamadora."
            )

        raise RuntimeError(f"Todos os modelos Gemini/Azure falharam. Ultimo erro: {last_exc}")

# ─────────────────────────────────────────────────────────────────────────────
# 🎤 WHISPER (stable-ts + faster-whisper)
# ─────────────────────────────────────────────────────────────────────────────
GPU_COUNT    = torch.cuda.device_count()
device       = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if torch.cuda.is_available() else "int8"

print(f"\nGPUs disponíveis: {GPU_COUNT}")
print(f"Carregando Whisper em {device}:0 ({COMPUTE_TYPE})...")
if False:
    print("ERRO: Caminho do Whisper nao encontrado!")
else:
    stable_model = stable_whisper.load_faster_whisper(
        MODEL_WHISPER_PATH,
        device=device,
        device_index=0,        # GPU0 explícito (ctranslate2 não aceita "cuda:0")
        compute_type=COMPUTE_TYPE,
    )
    print("Whisper carregado em GPU0!")

# Preenchido pela Cel8 após subir servidores OmniVoice
OMNIVOICE_PORTS = []

# ─────────────────────────────────────────────────────────────────────────────
# 📋 RESUMO DA CONFIGURACAO
# ─────────────────────────────────────────────────────────────────────────────
print(f"""
=================================================================
  CONFIGURACAO DO PIPELINE
=================================================================
  Velocidade alvo : {TARGET_MIN_SPEED}x - {TARGET_MAX_SPEED}x
  Fallback OpenAI : {USE_OPENAI_FALLBACK} ({OPENAI_FALLBACK_MODEL})
-----------------------------------------------------------------
  Identificacao   : {' > '.join(MODELS_IDENTIFICACAO)}
  Traducao        : {' > '.join(MODELS_TRADUCAO)}
  Reescrita       : {' > '.join(MODELS_REESCRITA)}
=================================================================
""")

print("SETUP CONCLUIDO! Execute a Fase 5 (Servidores OmniVoice) antes de rodar a Fabrica de Audio.")

cell_end(1, "done", "Celula 1 concluido")

In [ ]:
cell_start(10, "Montagem Final")
# @title 🎹 7. Montagem Final + SRT Híbrido
# Monta Vídeo Completo mesclando os 4 zips + JSONs do Omni Paralelo.

import os, gc, json, glob, zipfile
from pydub import AudioSegment
from tqdm.notebook import tqdm

BASE_PATH  = globals().get('BASE_PATH', '/kaggle/working/AUDIO_DUB')
TEMP_DIR   = globals().get('TEMP_PATH', f'{BASE_PATH}/TEMP')

current_anime_title = globals().get('selected_anime', 'Anime_Desconhecido')
safe_anime = current_anime_title.replace(' ', '_').replace('/', '_').replace("'", "").replace('"', '')

TARGET_DDBFS = -18.0

def normalize_segment(seg: AudioSegment, target: float = TARGET_DDBFS) -> AudioSegment:
    if seg.dBFS == float('-inf'):
        return seg
    return seg.apply_gain(target - seg.dBFS)

def sanitize_blocks(lista):
    lista = sorted(lista, key=lambda b: b['start'])
    out, cursor = [], 0.0
    for blk in lista:
        blk = dict(blk)
        if blk['start'] < cursor:
            blk['start'] = cursor
        if blk['start'] >= blk['end']:
            continue
        out.append(blk)
        cursor = blk['end']
    return out

def fmt_ts(s):
    ms = int((s - int(s)) * 1000)
    m, sec = divmod(int(s), 60)
    h, m = divmod(m, 60)
    return f"{h:02d}:{m:02d}:{sec:02d},{ms:03d}"

def run_montagem():
    modo = "Video Completo"
    prefix     = "block_"
    json_final = f"{BASE_PATH}/ROTEIRO_ADAPTADO_COMPLETO_FINAL.json"
    modo_folder = 'Completo'

    print(f"\n{'='*60}")
    print(f"  [ASSEMBLE] Baixando e mesclando os 4 zips de TTS paralelos...")
    print(f"{'='*60}")

    # 1. Baixar e descompactar os 4 zips de áudio dos TTS paralelos
    for i in range(1, 5):
        drive_zip = f"KAGGLE/AUDIO_DUB/omni_tts_pt{i}.zip"
        local_zip = f"{BASE_PATH}/omni_tts_pt{i}.zip"
        print(f"  Baixando {drive_zip}...")
        try:
            if 'baixar_do_drive' in globals():
                baixar_do_drive(drive_zip, local_zip)
            if os.path.exists(local_zip):
                with zipfile.ZipFile(local_zip, 'r') as zip_ref:
                    zip_ref.extractall(TEMP_DIR)
                print(f"  [OK] Descompactado {local_zip} em {TEMP_DIR}.")
        except Exception as e:
            print(f"  [AVISO] Falha ao baixar/descompactar zip {i}: {e}")

    # 2. Baixar e mesclar os 4 JSONs de roteiro adaptados
    merged_blocks = []
    for i in range(1, 5):
        drive_json = f"KAGGLE/AUDIO_DUB/ROTEIRO_ADAPTADO_COMPLETO_FINAL_pt{i}.json"
        local_json = f"{BASE_PATH}/ROTEIRO_ADAPTADO_COMPLETO_FINAL_pt{i}.json"
        print(f"  Baixando {drive_json}...")
        try:
            if 'baixar_do_drive' in globals():
                baixar_do_drive(drive_json, local_json)
            if os.path.exists(local_json):
                with open(local_json, 'r', encoding='utf-8') as f:
                    part_blocks = json.load(f)
                    for b in part_blocks:
                        b['part'] = i
                    merged_blocks.extend(part_blocks)
                print(f"  [OK] Carregado {len(part_blocks)} blocos da parte {i}.")
        except Exception as e:
            print(f"  [AVISO] Falha ao baixar/carregar JSON {i}: {e}")

    if merged_blocks:
        merged_blocks = sorted(merged_blocks, key=lambda x: x['start'])
        with open(json_final, 'w', encoding='utf-8') as f:
            json.dump(merged_blocks, f, ensure_ascii=False, indent=4)
        print(f"  [ASSEMBLE] Mesclados {len(merged_blocks)} blocos no {os.path.basename(json_final)}")
        lista_render = merged_blocks
    elif os.path.exists(json_final):
        with open(json_final, 'r', encoding='utf-8') as f:
            lista_render = json.load(f)
    elif os.path.exists(f"{BASE_PATH}/traducao_simplificada.json"):
        with open(f"{BASE_PATH}/traducao_simplificada.json", 'r', encoding='utf-8') as f:
            lista_render = json.load(f)
    else:
        print(f"  AVISO: Nenhum JSON encontrado. Pulando montagem.")
        return

    def find_block_audio(block):
        audio_file = block.get('current_file')
        if audio_file and os.path.exists(audio_file):
            return audio_file
        part_idx = block.get('part', 1)
        block_id = block['id']
        p1 = os.path.join(TEMP_DIR, f"pt{part_idx}_block_{block_id}_v*.mp3")
        p2 = os.path.join(TEMP_DIR, f"*pt{part_idx}*block_{block_id}_v*.mp3")
        arqs = glob.glob(p1) or glob.glob(p2)
        if arqs:
            return max(arqs, key=lambda x: int(x.split('_v')[-1].split('.mp3')[0]) if '_v' in x else 0)
        p3 = os.path.join(TEMP_DIR, f"*block_{block_id}_v*.mp3")
        p4 = os.path.join(TEMP_DIR, f"*{block_id}_v*.mp3")
        arqs = glob.glob(p3) or glob.glob(p4)
        if arqs:
            return max(arqs, key=lambda x: int(x.split('_v')[-1].split('.mp3')[0]) if '_v' in x else 0)
        return None

    FINAL_OUTPUT_DIR = f"{BASE_PATH}/{safe_anime}/{modo_folder}"
    os.makedirs(FINAL_OUTPUT_DIR, exist_ok=True)

    OUTPUT_FILENAME = f"{FINAL_OUTPUT_DIR}/VIDEO_COMPLETO_FINAL.mp3"
    SRT_FILENAME = OUTPUT_FILENAME.replace('.mp3', '.srt')

    lista_render = sanitize_blocks(lista_render)
    if not lista_render:
        print("  AVISO: Lista de blocos vazia apos sanitizacao.")
        return

    start_offset = lista_render[0]['start']

    # --- Concatenação Sequencial exata do Drama-pipeline ---
    print(f"  Montando {len(lista_render)} blocos...")
    segments_to_concat = []
    cursor_ms = 0
    actual_starts_ms = {}

    for b_idx, block in enumerate(tqdm(lista_render, desc=f'Mixing {modo[:5]}')):
        block_id = block['id']
        audio_file = find_block_audio(block)
        if not audio_file:
            print(f"  [AVISO] Audio nao encontrado para bloco {block_id} (pt{block.get('part', 1)})")
            continue

        seg   = AudioSegment.from_file(audio_file)
        seg   = normalize_segment(seg)

        ratio = block.get('speed_factor', 1.0)
        tipo  = block.get('tipo', 'NARRACAO')
        slot_duration_ms = int((block['end'] - block['start']) * 1000)
        dur_antes = len(seg)

        if tipo == 'NARRACAO' and ratio != 1.0:
            safe_speed = max(0.5, min(ratio, 3.0))
            tmp_wav  = f"tmp_mix_{b_idx}.wav"
            proc_wav = f"proc_mix_{b_idx}.wav"
            try:
                seg.export(tmp_wav, format="wav")
                cmd = [
                    'ffmpeg', '-y', '-i', tmp_wav,
                    '-filter:a', f'atempo={safe_speed}',
                    '-ar', '44100', proc_wav
                ]
                result = subprocess.run(
                    cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE,
                    capture_output=True
                )
                if result.returncode == 0 and os.path.exists(proc_wav):
                    seg = AudioSegment.from_wav(proc_wav)
                    os.remove(proc_wav)
            except Exception as e:
                print(f'  [ERRO atempo] bloco {block_id}: {e}')
            finally:
                if os.path.exists(tmp_wav): os.remove(tmp_wav)

        if slot_duration_ms > 0 and len(seg) > slot_duration_ms + 200:
            seg = seg[:slot_duration_ms]

        block_start_ms = int((block['start'] - start_offset) * 1000)
        gap = block_start_ms - cursor_ms
        if gap > 10:
            gap = min(gap, 2000)
            segments_to_concat.append(AudioSegment.silent(duration=gap))
            cursor_ms += gap

        actual_starts_ms[b_idx] = cursor_ms
        segments_to_concat.append(seg)
        cursor_ms += len(seg)

    if not segments_to_concat:
        print("  AVISO: Nenhum áudio montado.")
        return

    print(f"  Concatenando segmentos...")
    final_mix = segments_to_concat[0]
    for s in segments_to_concat[1:]:
        final_mix = final_mix + s

    if globals().get('PROJECT_OPTS', {}).get('bg_audio'):
        bg_audio_path = "/kaggle/working/demucs_out/htdemucs/anime_audio/no_vocals.wav"
        if os.path.exists(bg_audio_path):
            print("  [BGM] Mixando áudio instrumental de fundo...")
            bgm = AudioSegment.from_file(bg_audio_path)
            if len(bgm) < len(final_mix):
                bgm = bgm + AudioSegment.silent(duration=len(final_mix) - len(bgm))
            elif len(bgm) > len(final_mix):
                bgm = bgm[:len(final_mix)]
            bgm = bgm.apply_gain(-5.0)
            final_mix = bgm.overlay(final_mix)

    audio_duration_seconds = len(final_mix) / 1000.0
    print(f'  Exportando: {OUTPUT_FILENAME}')
    final_mix.export(OUTPUT_FILENAME, format='mp3')
    del final_mix
    gc.collect()

    # --- SRT Híbrido ---
    print(f'  Gerando SRT...')
    srt_entries = []
    for b_idx, block in enumerate(tqdm(lista_render, desc='SRT')):
        tipo     = block.get('tipo', 'NARRACAO')
        block_id = block['id']
        if tipo == 'NARRACAO':
            audio_file = find_block_audio(block)
            if not audio_file: continue
            try:
                if 'stable_model' in globals() and stable_model is not None:
                    result = stable_model.transcribe_stable(audio_file, language='pt', word_timestamps=True)
                    time_offset = actual_starts_ms.get(b_idx, int((block['start'] - start_offset) * 1000)) / 1000.0
                    all_words = []
                    ratio = block.get('speed_factor', 1.0)
                    safe_speed = max(0.5, min(ratio, 3.0)) if ratio != 1.0 else 1.0

                    for seg in result.segments:
                        for w in seg.words:
                            all_words.append({'word': w.word.strip(), 'start': w.start / safe_speed, 'end': w.end / safe_speed})

                    joined = []
                    for w in all_words:
                        if w['word'].startswith('-') and joined:
                            joined[-1]['word'] += w['word']
                            joined[-1]['end'] = w['end']
                        else:
                            joined.append(dict(w))
                    all_words = joined

                    MAX_CHARS_LINE = 50

                    def format_sub(wlist):
                        full = " ".join(ww['word'] for ww in wlist)
                        if len(full) <= MAX_CHARS_LINE:
                            return full
                        mid = len(full) // 2
                        best_pos = -1
                        best_score = 9999
                        pos = 0
                        for wi in range(len(wlist) - 1):
                            pos += len(wlist[wi]['word'])
                            score = abs(pos - mid)
                            if wlist[wi]['word'].endswith(','):
                                score -= 20
                            if score < best_score and wi > 0:
                                best_score = score
                                best_pos = pos
                            pos += 1
                        if best_pos <= 0:
                            return full
                        line1 = full[:best_pos].rstrip()
                        line2 = full[best_pos:].lstrip()
                        return f"{line1}\n{line2}"

                    is_word_by_word = globals().get('PROJECT_OPTS', {}).get('srt_type') == 'word_by_word'

                    idx = 0
                    curr_block = []
                    while idx < len(all_words):
                        w = all_words[idx]
                        if is_word_by_word:
                            srt_entries.append({
                                'block_idx': b_idx,
                                'start': time_offset + w['start'],
                                'end':   time_offset + w['end'],
                                'text':  w['word'].strip()
                            })
                            idx += 1
                            continue

                        curr_block.append(w)
                        has_strong = any(p in w['word'] for p in ['.', '?', '!'])
                        has_comma = ',' in w['word']

                        cut = False
                        if has_strong:
                            cut = True
                        elif len(curr_block) >= 12:
                            cut = True
                        elif has_comma and len(curr_block) >= 5:
                            lookahead = 0
                            for i in range(idx + 1, len(all_words)):
                                lookahead += 1
                                lw = all_words[i]['word']
                                if any(p in lw for p in ['.', '?', '!', ',']):
                                    break
                            if lookahead > 5:
                                cut = True
                            elif len(curr_block) + lookahead > 10:
                                cut = True

                        if cut:
                            text = format_sub(curr_block)
                            srt_entries.append({
                                'block_idx': b_idx,
                                'start': time_offset + curr_block[0]['start'],
                                'end':   time_offset + curr_block[-1]['end'],
                                'text':  text
                            })
                            curr_block = []

                        idx += 1

                    if curr_block and not is_word_by_word:
                        text = format_sub(curr_block)
                        srt_entries.append({
                            'block_idx': b_idx,
                            'start': time_offset + curr_block[0]['start'],
                            'end':   time_offset + curr_block[-1]['end'],
                            'text':  text
                        })
                else:
                    raise Exception("stable_model nao disponivel.")
            except Exception:
                real_start = actual_starts_ms.get(block_id, int((block['start'] - start_offset) * 1000)) / 1000.0
                srt_entries.append({'block_idx': b_idx, 
                    'start': real_start,
                    'end':   real_start + (block['end'] - block['start']),
                    'text':  block.get('translated_text', '')
                })
        elif tipo == 'CENA' and block.get('translated_text', '').strip():
            real_start = actual_starts_ms.get(block_id, int((block['start'] - start_offset) * 1000)) / 1000.0
            srt_entries.append({'block_idx': b_idx, 
                'start': real_start,
                'end':   real_start + (block['end'] - block['start']),
                'text':  block['translated_text']
            })

    srt_entries.sort(key=lambda x: (x.get('block_idx', 0), x['start']))
    cleaned = []
    last_end = -1.0
    for entry in srt_entries:
        if entry['start'] < last_end:
            entry['start'] = last_end
        if entry['start'] >= audio_duration_seconds:
            continue
        if entry['end'] <= entry['start']:
            entry['end'] = entry['start'] + 0.5
        if entry['end'] > audio_duration_seconds:
            entry['end'] = audio_duration_seconds
        if entry['end'] <= entry['start']:
            continue
        cleaned.append(entry)
        last_end = entry['end']
    srt_entries = cleaned
    with open(SRT_FILENAME, 'w', encoding='utf-8') as srt_file:
        for idx, entry in enumerate(srt_entries, 1):
            srt_file.write(f"{idx}\n")
            srt_file.write(f"{fmt_ts(entry['start'])} --> {fmt_ts(entry['end'])}\n")
            srt_file.write(f"{entry['text']}\n\n")

    print(f'  SRT: {len(srt_entries)} entradas salvas.')

    if 'salvar_no_drive' in globals():
        drive_mp3 = f"KAGGLE/AUDIO_DUB/OUTPUT/{safe_anime}_{modo_folder}.mp3"
        drive_srt = f"KAGGLE/AUDIO_DUB/OUTPUT/{safe_anime}_{modo_folder}.srt"
        try:
            salvar_no_drive(OUTPUT_FILENAME, drive_mp3)
            salvar_no_drive(SRT_FILENAME,    drive_srt)
            print(f"  Backup Drive: {drive_mp3}")

            intro_mp3_local = os.path.join(TEMP_DIR, "block_intro.mp3")
            if os.path.exists(intro_mp3_local):
                salvar_no_drive(intro_mp3_local, "KAGGLE/AUDIO_DUB/intro_audio.mp3")
                salvar_no_drive(intro_mp3_local, "KAGGLE/PIPELINE/OMNI/intro_audio.mp3")
                try:
                    if 'stable_model' in globals() and stable_model is not None:
                        intro_res = stable_model.transcribe_stable(intro_mp3_local, language='pt', word_timestamps=True)
                        intro_words = []
                        for seg in intro_res.segments:
                            for w in seg.words:
                                intro_words.append({'word': w.word.strip(), 'start': w.start, 'end': w.end})
                        intro_srt_file = os.path.join(TEMP_DIR, "intro_output.srt")
                        with open(intro_srt_file, 'w', encoding='utf-8') as f_intro_srt:
                            for idx_i, entry_i in enumerate(intro_words, 1):
                                f_intro_srt.write(f"{idx_i}\n")
                                f_intro_srt.write(f"{fmt_ts(entry_i['start'])} --> {fmt_ts(entry_i['end'])}\n")
                                f_intro_srt.write(f"{entry_i['word']}\n\n")
                        salvar_no_drive(intro_srt_file, "KAGGLE/AUDIO_DUB/intro_output.srt")
                        salvar_no_drive(intro_srt_file, "KAGGLE/PIPELINE/OMNI/intro_output.srt")
                        print("  [INTRO SRT] SRT da introdução sincronizado enviado ao Drive com sucesso!")
                except Exception as e_intro_srt:
                    print("  [INTRO SRT AVISO] Falha ao transcrever intro:", e_intro_srt)
        except Exception as e:
            print(f"  Erro backup Drive: {e}")

# ─────────────────────────────────────────────────────────────────────────────
# EXECUÇÃO
# ─────────────────────────────────────────────────────────────────────────────
if should_run("assemble"):
    run_montagem()
    print("\nMONTAGEM FINALIZADA!")
    cell_end(10, "done", "Montagem concluída")
    report_step("done", "Processamento Omni completo finalizado com sucesso.")
else:
    print("Assemble não é necessário para esta task. Ignorando execução.")
    cell_end(10, "done", "Montador ignorado")


In [ ]:
cell_start(11, "Busca Audio Clonagem")
# @title 🧹 8. Faxina Cirúrgica (Painel de Controle)
# @markdown Selecione exatamente o que você deseja remover para liberar espaço ou refazer etapas.

import os
import shutil
import glob

# --- PAINEL DE CONTROLE ---
DELETE_TRANSCRIPTION = True # @param {type:"boolean"}
DELETE_TRANSLATION = True # @param {type:"boolean"}
DELETE_IDENTIFICATION = True # @param {type:"boolean"}
DELETE_SHORT_DATA = True # @param {type:"boolean"}
DELETE_AUDIO_CACHE = True # @param {type:"boolean"}
DELETE_FINAL_FILES = True # @param {type:"boolean"}

print("🧹 Iniciando Protocolo de Limpeza Seletiva...")

BASE_PATH = globals().get('BASE_PATH', '/kaggle/working/AUDIO_DUB')

# 1. Definição dos Alvos com BASE_PATH correto
targets = {
    "transcription": [f"{BASE_PATH}/transcricao_raw.json"],
    "translation":   [f"{BASE_PATH}/traducao_simplificada.json",
                      f"{BASE_PATH}/traducao_feita.json"],
    "identification":[f"{BASE_PATH}/identificacao_anime.json"],
    "short_data":    [f"{BASE_PATH}/GUIA_VIRAL_*.txt",
                      f"{BASE_PATH}/GUIA_POSTAGEM_*.txt",
                      f"{BASE_PATH}/roteiro_short.json",
                      f"{BASE_PATH}/ROTEIRO_ADAPTADO_SHORT_FINAL.json"],
    "final_files":   [f"{BASE_PATH}/**/SHORT_VIRAL_FINAL.mp3",
                      f"{BASE_PATH}/**/VIDEO_COMPLETO_FINAL.mp3",
                      f"{BASE_PATH}/**/DUB_FINAL_*.mp3",
                      f"{BASE_PATH}/ROTEIRO_ADAPTADO_COMPLETO_FINAL.json"],
}

# 2. Contador de Remoções
deleted_count = 0

# --- FUNÇÃO DE REMOÇÃO ---
def remove_pattern(pattern_list):
    count = 0
    for pat in pattern_list:
        # recursive=True para pegar arquivos em subpastas (**/)
        for f in glob.glob(pat, recursive=True):
            try:
                os.remove(f)
                count += 1
            except Exception as e:
                print(f"   Erro ao apagar {f}: {e}")
    return count

# --- EXECUÇÃO ---

# 1. Transcrição (Raw)
if DELETE_TRANSCRIPTION:
    n = remove_pattern(targets["transcription"])
    if n > 0: print(f"🗑️ Transcrição Original removida ({n} arquivos).")

# 2. Tradução (Base)
if DELETE_TRANSLATION:
    n = remove_pattern(targets["translation"])
    if n > 0: print(f"🗑️ Tradução Base removida ({n} arquivos).")

# 3. Identificação do Anime
if DELETE_IDENTIFICATION:
    n = remove_pattern(targets["identification"])
    if n > 0: print(f"🗑️ Identificação do anime removida ({n} arquivos).")

# 4. Dados do Short e Guias
if DELETE_SHORT_DATA:
    n = remove_pattern(targets["short_data"])
    # Resetar variáveis de memória também é importante
    if 'short_blocks' in globals():
        short_blocks = []
        print("   🧠 Memória 'short_blocks' limpa.")
    if n > 0: print(f"🗑️ Guias e Shorts removidos ({n} arquivos).")

# 5. Arquivos Finais (MP3 Prontos)
if DELETE_FINAL_FILES:
    n = remove_pattern(targets["final_files"])
    if n > 0: print(f"🗑️ Arquivos Finais (.mp3) removidos ({n} arquivos).")

# 6. Cache de Áudio (Pasta Temporária)
if DELETE_AUDIO_CACHE:
    temp_dir = globals().get('TEMP_PATH', '/kaggle/working/temp_audio')
    demucs_dir = "/kaggle/working/demucs_out"
    for d in [temp_dir, demucs_dir]:
        if os.path.exists(d):
            try:
                shutil.rmtree(d)
                os.makedirs(d, exist_ok=True)
                print(f"🗑️ {d} limpo e recriado.")
            except Exception as e:
                print(f"❌ Erro na pasta {d}: {e}")

# Resumo Final
print("-" * 40)
print("✨ Limpeza concluída conforme solicitado.")

cell_end(11, "done", "Busca Audio Clonagem concluido")
report_step("done", "Processamento Omni finalizado")
